# Nortek Aquadopp Data Processing

Processing path used: 

build_aqd_dataset_from_metadata -> trim deployment window -> IMOSNetCDFConverter_AQD

### Setup

Imports

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import numpy as np
import pandas as pd

Import local tools

In [ ]:
TOOLS_DIR = Path.cwd().resolve().parent
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

# For IMOS NetCDF conversion
from tools.imos_nc_converter import imos_converter as imos_converter_module
importlib.reload(imos_converter_module)
IMOSNetCDFConverter_AQD = imos_converter_module.IMOSNetCDFConverter_AQD

# Read the metadata table
from tools import database_lookup as database_lookup_module
importlib.reload(database_lookup_module)
get_instrument_context = database_lookup_module.get_instrument_context
update_metadata_file_fields = database_lookup_module.update_metadata_file_fields

# Generic helpers
from tools.helpers import plot_data_by_qc

# QC flagging
from tools.helpers import apply_qc_flag_windows
from tools.helpers.manual_qc_flags import build_deployment_good_data_windows

# Aquadopp parser
from tools.parsers.read_aqd import build_aqd_dataset_from_metadata

Definitions

In [ ]:
# Working directory
os.chdir("/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/ash")

In [ ]:
# Select instrument using ID from satellite_altimetry_moorings_metadata.csv

inst_deploy_id = 258    # rec_202502/BASS3A_PTSUV

database, _row, cfg, metadata = get_instrument_context(
    inst_deploy_id=inst_deploy_id,
    # metadata_csv="/datasets/work/oa-srsalt/work/preqa/SWOT/cal_val/jason_calval/all_mooring_data/reference_mooring_proc_info/satellite_altimetry_moorings_metadata.csv",
    print_details=True,
)

In [ ]:
converter = IMOSNetCDFConverter_AQD(input_folder="", input_file="", output_dir="")

### Proc_1


Read and create ds, df

In [ ]:
# Read AQD text file from metadata-defined input location and build dataframe rows.
df, ds, dat_file, range_tag = build_aqd_dataset_from_metadata(_row, cwd=Path.cwd(), verbose=True)

# Keep TIME as naive datetime (no timezone complications)
ds['TIME'] = pd.to_datetime(ds['TIME'].values)

In [ ]:
PLOT_VARS = None  # set None for defaults, to specify use ["UCUR", "VCUR", "TEMP", "DEPTH", "CNDC", "PSAL", "PRES"] 
fig = plot_data_by_qc(
    ds,
    variables=PLOT_VARS,
    flags_to_plot=[1],      # optional
    y_zoom_to_good=True,    # optional
)
fig.show()

In [ ]:
# Get time coverage from metadata and strip timezone
time_coverage_start = pd.to_datetime(_row.get("time_coverage_start")).tz_localize(None)
time_coverage_end = pd.to_datetime(_row.get("time_coverage_end")).tz_localize(None)

print(f"time_coverage_start: {time_coverage_start}, time_coverage_end: {time_coverage_end}")

# Apply QC: flag data OUTSIDE the deployment window as bad (flag=4)
deployment_window = [{
    "qc_vars": ["TEMP_quality_control", "UCUR_quality_control", "VCUR_quality_control", "DEPTH_quality_control"],
    "flag": 4,
    "start": str(time_coverage_start),
    "end": str(time_coverage_end),
    "comment": "outside deployment window"
}]

ds_proc1 = apply_qc_flag_windows(ds, deployment_window, time_name="TIME", verbose=False)

In [ ]:
# Trim to deployment window
ds_proc1_trimmed = ds_proc1.sel(TIME=slice(time_coverage_start, time_coverage_end))

# Convert TIME to numeric days since 1950-01-01
time_pd = pd.to_datetime(ds_proc1_trimmed['TIME'].values)
reference_date = pd.Timestamp('1950-01-01')
time_numeric = (time_pd - reference_date).total_seconds() / (86400.0)


Write trimmed file to proc_1

In [ ]:
# Create NetCDF
proc_1_out = converter.process(
    time_data=time_numeric,
    temp_data=ds_proc1_trimmed['TEMP'].values,
    temp_qc_data=ds_proc1_trimmed['TEMP_quality_control'].values,
    ucur_data=ds_proc1_trimmed['UCUR'].values,
    ucur_qc_data=ds_proc1_trimmed['UCUR_quality_control'].values,
    vcur_data=ds_proc1_trimmed['VCUR'].values,
    vcur_qc_data=ds_proc1_trimmed['VCUR_quality_control'].values,
    depth_data=ds_proc1_trimmed['DEPTH'].values if 'DEPTH' in ds_proc1_trimmed else None,
    depth_qc_data=ds_proc1_trimmed['DEPTH_quality_control'].values if 'DEPTH_quality_control' in ds_proc1_trimmed else None,
    longitude=float(cfg["longitude"]),
    latitude=float(cfg["latitude"]),
    depth=float(cfg.get("nominal_depth", 0.0)),
    inst_channels=str(cfg.get("inst_channels", "")),
    start_of_good_data=time_coverage_start,
    site_code=str(cfg["location"]),
    version=str(cfg.get("version", "1")),
    instrument=str(cfg.get("inst_type", "")),
    inst_id=str(int(cfg["inst_id"])),
    location=cfg["location"],
    output_name_mode="internal",
    output_stage="proc_1",
    metadata_row=_row,
    metadata_mode="fill_missing",
    deployment_id=cfg.get("deployment_id", ""),
    mooring_channels=cfg.get("mooring_channels", ""),
    nominal_inst_depth=cfg.get("nominal_inst_depth", ""),
    time_coverage_start=time_coverage_start,
    time_coverage_end=time_coverage_end,
)

Write proc_1 filename to metadata table

In [ ]:
proc_1_name = Path(proc_1_out).name
_row = update_metadata_file_fields(inst_deploy_id, {"proc_1_file": proc_1_name}, output_paths={"proc_1_file": proc_1_out}, working_dir=Path.cwd())
print(f"Updated proc_1_file: {_row['proc_1_file']}")